In [1]:
# Import all files needed
import pandas as pd
df_uber = pd.read_csv("C:/Users/YawOM/OneDrive/Desktop/Data Science/Takeaway project/ubereats_orders.csv")
df_instore = pd.read_csv("C:/Users/YawOM/OneDrive/Desktop/Data Science/Takeaway project/instore_till.csv")
df_roo = pd.read_csv("C:/Users/YawOM/OneDrive/Desktop/Data Science/Takeaway project/deliveroo_orders.csv")

In [2]:
# Gather critical information for the project, data types, name of columns, etc
df_uber.head(3)

,order_ref,date_time,product_title,qty,subtotal
0,UBR-503718,02/05/2026 23:02,soft drink,2,5.0
1,UBR-662335,07/05/2026 16:51,chocolate milkshake,1,4.5
2,UBR-494939,28/04/2026 16:21,soft drink,1,2.5


In [3]:
df_uber.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_ref      100 non-null    object 
 1   date_time      100 non-null    object 
 2   product_title  100 non-null    object 
 3   qty            100 non-null    int64  
 4   subtotal       100 non-null    float64
dtypes: float64(1), int64(1), object(3)
memory usage: 4.0+ KB


In [4]:
df_roo.head(3)

,Order Reference,Order Date & Time,Item Description,Quantity Ordered,Gross Amount
0,DLV-5073,"May 15, 2026, 11:40 AM",Sweet Potato Fries,2,8.0
1,DLV-5340,"Apr 24, 2026, 02:02 PM",Vegan Burger,1,9.0
2,DLV-8548,"May 17, 2026, 08:20 PM",Soft Drink,2,5.0


In [5]:
df_roo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Order Reference    100 non-null    object 
 1   Order Date & Time  100 non-null    object 
 2   Item Description   100 non-null    object 
 3   Quantity Ordered   100 non-null    int64  
 4   Gross Amount       100 non-null    float64
dtypes: float64(1), int64(1), object(3)
memory usage: 4.0+ KB


In [6]:
df_instore.head(3)

,Order_ID,Timestamp,Item_Name,Quantity,Price_Paid
0,TXN-1001,2026-05-14 11:35:26,Regular Fries,3,9.0
1,TXN-1002,2026-04-24 23:20:45,Gourmet Burger,1,8.5
2,TXN-1003,2026-04-29 16:09:51,Regular Fries,1,3.0


In [7]:
df_instore.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Order_ID    100 non-null    object 
 1   Timestamp   100 non-null    object 
 2   Item_Name   100 non-null    object 
 3   Quantity    100 non-null    int64  
 4   Price_Paid  100 non-null    float64
dtypes: float64(1), int64(1), object(3)
memory usage: 4.0+ KB


In [8]:
# Convert all time entries into a datetime data type, crucial for the dashboard

df_roo["Order Date & Time"] = pd.to_datetime(df_roo["Order Date & Time"], format = "%b %d, %Y, %I:%M %p")
df_uber["date_time"] = pd.to_datetime(df_uber["date_time"], format = "%d/%m/%Y %H:%M")
df_instore["Timestamp"] = pd.to_datetime(df_instore["Timestamp"], format = "%Y-%m-%d %H:%M:%S")

In [9]:
# Add a distinguishing column identifying where the orders came from

df_roo["Source"] = "Deliveroo"
df_uber["Source"] = "Uber Eats"
df_instore["Source"] = "Till"

In [10]:
# This is an estimate on what the profit taken in by the business could realistically look like, depending on the platform or method of purchase.

df_uber["Estimated_Profit"] = df_uber["subtotal"] * 0.7
df_roo["Estimated_Profit"] = df_roo["Gross Amount"] * 0.75
df_instore["Estimated_Profit"] = df_instore["Price_Paid"] * 0.98

In [11]:
# Give all columns a common name so merging/combining data becomes easier

df_uber = df_uber.rename(columns = {"order_ref" : "Order_ID", 
                     "date_time" : "Timestamp", 
                     "product_title" :	"Item_Name",
                     "qty" : "Quantity",
                     "subtotal" : "Price_Paid"})
df_roo = df_roo.rename(columns = {"Order Reference" : "Order_ID", 
                     "Order Date & Time" : "Timestamp", 
                     "Item Description" :	"Item_Name",
                     "Quantity Ordered" : "Quantity",
                     "Gross Amount" : "Price_Paid"})

In [12]:
# Below identify all the unique entries in each file for the products

df_uber["Item_Name"].value_counts()

Item_Name
gourmet burger         18
cheeseburger           16
soft drink             15
regular fries          14
chocolate milkshake    13
sweet potato fries     13
vegan burger           11
Name: count, dtype: int64

In [13]:
df_roo["Item_Name"].value_counts()

Item_Name
Regular Fries          22
Cheeseburger           18
Soft Drink             16
Chocolate Milkshake    14
Vegan Burger           12
Sweet Potato Fries     10
Gourmet Burger          8
Name: count, dtype: int64

In [14]:
df_instore["Item_Name"].value_counts()

Item_Name
Chocolate Milkshake    18
Vegan Burger           18
Soft Drink             16
Gourmet Burger         14
Regular Fries          14
Sweet Potato Fries     11
Cheeseburger            9
Name: count, dtype: int64

In [15]:
# Again keep all data entries common between files

df_roo["Item_Name"] = df_roo["Item_Name"].str.lower()
df_instore["Item_Name"] = df_instore["Item_Name"].str.lower()

In [16]:
# Make the "master" dataset containing all the order combined from each platform/method of purchase

df_master = pd.concat([
    df_roo[["Order_ID", "Timestamp", "Item_Name", "Quantity", "Price_Paid", "Source", "Estimated_Profit"]], 
    df_instore[["Order_ID", "Timestamp", "Item_Name", "Quantity", "Price_Paid", "Source", "Estimated_Profit"]], 
    df_uber[["Order_ID", "Timestamp", "Item_Name", "Quantity", "Price_Paid", "Source", "Estimated_Profit"]]
], axis=0, ignore_index=True)

In [17]:
# Sanity check, check the contents of the master dataset

df_master

,Order_ID,Timestamp,Item_Name,Quantity,Price_Paid,Source,Estimated_Profit
0,DLV-5073,2026-05-15 11:40:00,sweet potato fries,2,8.0,Deliveroo,6.00
1,DLV-5340,2026-04-24 14:02:00,vegan burger,1,9.0,Deliveroo,6.75
2,DLV-8548,2026-05-17 20:20:00,soft drink,2,5.0,Deliveroo,3.75
3,DLV-6377,2026-04-26 17:01:00,sweet potato fries,1,4.0,Deliveroo,3.00
4,DLV-6586,2026-04-24 18:35:00,chocolate milkshake,2,9.0,Deliveroo,6.75
...,...,...,...,...,...,...,...
295,UBR-448999,2026-05-11 23:48:00,chocolate milkshake,2,9.0,Uber Eats,6.30
296,UBR-927825,2026-05-17 19:06:00,soft drink,2,5.0,Uber Eats,3.50
297,UBR-724457,2026-05-19 21:55:00,cheeseburger,2,19.0,Uber Eats,13.30
298,UBR-469036,2026-04-23 14:11:00,cheeseburger,1,9.5,Uber Eats,6.65


In [18]:
# Save file to computer

df_master.to_csv("C:/Users/YawOM/OneDrive/Desktop/Data Science/Takeaway project/final_takeaway_sales.csv", index=False)